# Step 01_03_03 -- Table Utility Assessment

**Phase:** 01 -- Data Exploration
**Pipeline Section:** 01_03 -- Systematic Data Profiling
**Dataset:** sc2egset
**Question:** Which of the 6 raw data objects (replay_players_raw,
replays_meta_raw, map_aliases_raw, game_events_raw, tracker_events_raw,
message_events_raw) are necessary and sufficient for the prediction
pipeline? What join key connects the two core tables? Are map names
already in English? What do loop=0 events represent?
**Invariants applied:** #3 (I3 temporal classification), #6 (all SQL
stored verbatim), #9 (profiling only -- no cleaning decisions)
**Predecessor:** 01_03_02 (True 1v1 Match Identification -- complete)
**Type:** Read-only -- no DuckDB writes

**Commit:** feat/table-utility-assessment
**Date:** 2026-04-16
**ROADMAP ref:** 01_03_03

## Cell 1 -- Imports

In [1]:
import json
from pathlib import Path

import pandas as pd

from rts_predict.common.notebook_utils import (
    get_notebook_db,
    get_reports_dir,
    setup_notebook_logging,
)

logger = setup_notebook_logging()

## Cell 2 -- DuckDB connection and output directories

In [2]:
conn = get_notebook_db("sc2", "sc2egset")
reports_dir = get_reports_dir("sc2", "sc2egset")

artifact_dir = (
    reports_dir / "artifacts" / "01_exploration" / "03_profiling"
)
artifact_dir.mkdir(parents=True, exist_ok=True)
print(f"Artifact dir: {artifact_dir}")

# Load prior census and profile artifacts for runtime constants (I7)
census_json_path = (
    reports_dir / "artifacts" / "01_exploration" / "02_eda"
    / "01_02_04_univariate_census.json"
)
with open(census_json_path) as f:
    census = json.load(f)

profile_json_path = (
    reports_dir / "artifacts" / "01_exploration" / "03_profiling"
    / "01_03_01_systematic_profile.json"
)
with open(profile_json_path) as f:
    profile = json.load(f)

true_1v1_json_path = (
    reports_dir / "artifacts" / "01_exploration" / "03_profiling"
    / "01_03_02_true_1v1_profile.json"
)
with open(true_1v1_json_path) as f:
    true_1v1 = json.load(f)

# Constants from prior artifacts (I7: no magic numbers)
RP_TOTAL_ROWS = census["null_census"]["replay_players_raw"]["total_rows"]
RM_TOTAL_ROWS = census["null_census"]["replays_meta_raw_filename"]["total_rows"]
TRUE_1V1_DECISIVE = true_1v1["replay_classification"]["true_1v1_decisive_count"]

print(f"replay_players_raw total rows (from census): {RP_TOTAL_ROWS}")
print(f"replays_meta_raw total rows (from census):   {RM_TOTAL_ROWS}")
print(f"true_1v1_decisive replays (from 01_03_02):   {TRUE_1V1_DECISIVE}")

sql_queries: dict = {}

Artifact dir: /Users/tomaszpionka/Projects/rts-outcome-prediction/src/rts_predict/games/sc2/datasets/sc2egset/reports/artifacts/01_exploration/03_profiling
replay_players_raw total rows (from census): 44817
replays_meta_raw total rows (from census):   22390
true_1v1_decisive replays (from 01_03_02):   22366


## Cell 3 -- T01: DESCRIBE both core tables

In [3]:
# DESCRIBE replay_players_raw
sql_queries["describe_replay_players_raw"] = "DESCRIBE replay_players_raw"
rp_schema_df = conn.con.execute(sql_queries["describe_replay_players_raw"]).df()
print("=== replay_players_raw schema ===")
print(rp_schema_df.to_string(index=False))
print(f"\nColumn count: {len(rp_schema_df)}")

=== replay_players_raw schema ===
        column_name column_type null  key default extra
           filename     VARCHAR   NO None    None  None
            toon_id     VARCHAR   NO None    None  None
           nickname     VARCHAR  YES None    None  None
           playerID     INTEGER  YES None    None  None
             userID      BIGINT  YES None    None  None
           isInClan     BOOLEAN  YES None    None  None
            clanTag     VARCHAR  YES None    None  None
                MMR     INTEGER  YES None    None  None
               race     VARCHAR  YES None    None  None
       selectedRace     VARCHAR  YES None    None  None
           handicap     INTEGER  YES None    None  None
             region     VARCHAR  YES None    None  None
              realm     VARCHAR  YES None    None  None
      highestLeague     VARCHAR  YES None    None  None
             result     VARCHAR  YES None    None  None
                APM     INTEGER  YES None    None  None
              

## Cell 4 -- T01: DESCRIBE replays_meta_raw

In [4]:
sql_queries["describe_replays_meta_raw"] = "DESCRIBE replays_meta_raw"
rm_schema_df = conn.con.execute(sql_queries["describe_replays_meta_raw"]).df()
print("=== replays_meta_raw schema ===")
print(rm_schema_df.to_string(index=False))
print(f"\nColumn count: {len(rm_schema_df)}")

=== replays_meta_raw schema ===
      column_name                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            column_type null  key default extra
         filename                                                                                                                                                                                                                                                                                                                                                                                                      

## Cell 5 -- T01: Full struct leaf field enumeration (all 31 fields)

replays_meta_raw has 9 top-level columns. The 4 STRUCT columns flatten
into 31 leaf fields:
- details (3 leaves): gameSpeed, isBlizzardMap, timeUTC
- header (2 leaves): elapsedGameLoops, version
- initData.gameDescription (22 leaves):
    gameOptions sub-struct (15 leaves):
      advancedSharedControl, amm, battleNet, clientDebugFlags,
      competitive, cooperative, fog, heroDuplicatesAllowed,
      lockTeams, noVictoryOrDefeat, observers, practice,
      randomRaces, teamsTogether, userDifficulty
    gameSpeed, isBlizzardMap, mapAuthorName, mapFileSyncChecksum,
    mapSizeX, mapSizeY, maxPlayers (7 leaves)
- metadata (4 leaves): baseBuild, dataBuild, gameVersion, mapName
Plus non-struct top-level: filename, ToonPlayerDescMap,
  gameEventsErr, messageEventsErr, trackerEvtsErr (5 non-struct)

In [5]:
sql_queries["struct_leaf_census"] = """
SELECT
    -- details struct (3 leaves)
    details.gameSpeed               AS details_gameSpeed,
    details.isBlizzardMap           AS details_isBlizzardMap,
    details.timeUTC                 AS details_timeUTC,

    -- header struct (2 leaves)
    header.elapsedGameLoops         AS header_elapsedGameLoops,
    header.version                  AS header_version,

    -- initData.gameDescription.gameOptions sub-struct (15 leaves)
    initData.gameDescription.gameOptions.advancedSharedControl
                                    AS go_advancedSharedControl,
    initData.gameDescription.gameOptions.amm
                                    AS go_amm,
    initData.gameDescription.gameOptions.battleNet
                                    AS go_battleNet,
    initData.gameDescription.gameOptions.clientDebugFlags
                                    AS go_clientDebugFlags,
    initData.gameDescription.gameOptions.competitive
                                    AS go_competitive,
    initData.gameDescription.gameOptions.cooperative
                                    AS go_cooperative,
    initData.gameDescription.gameOptions.fog
                                    AS go_fog,
    initData.gameDescription.gameOptions.heroDuplicatesAllowed
                                    AS go_heroDuplicatesAllowed,
    initData.gameDescription.gameOptions.lockTeams
                                    AS go_lockTeams,
    initData.gameDescription.gameOptions.noVictoryOrDefeat
                                    AS go_noVictoryOrDefeat,
    initData.gameDescription.gameOptions.observers
                                    AS go_observers,
    initData.gameDescription.gameOptions.practice
                                    AS go_practice,
    initData.gameDescription.gameOptions.randomRaces
                                    AS go_randomRaces,
    initData.gameDescription.gameOptions.teamsTogether
                                    AS go_teamsTogether,
    initData.gameDescription.gameOptions.userDifficulty
                                    AS go_userDifficulty,

    -- initData.gameDescription direct fields (7 leaves)
    initData.gameDescription.gameSpeed
                                    AS gd_gameSpeed,
    initData.gameDescription.isBlizzardMap
                                    AS gd_isBlizzardMap,
    initData.gameDescription.mapAuthorName
                                    AS gd_mapAuthorName,
    initData.gameDescription.mapFileSyncChecksum
                                    AS gd_mapFileSyncChecksum,
    initData.gameDescription.mapSizeX
                                    AS gd_mapSizeX,
    initData.gameDescription.mapSizeY
                                    AS gd_mapSizeY,
    initData.gameDescription.maxPlayers
                                    AS gd_maxPlayers,

    -- metadata struct (4 leaves)
    metadata.baseBuild              AS metadata_baseBuild,
    metadata.dataBuild              AS metadata_dataBuild,
    metadata.gameVersion            AS metadata_gameVersion,
    metadata.mapName                AS metadata_mapName

FROM replays_meta_raw
LIMIT 5
"""

struct_sample_df = conn.con.execute(sql_queries["struct_leaf_census"]).df()
print(f"Struct leaf sample shape: {struct_sample_df.shape}")
print(f"Columns extracted: {list(struct_sample_df.columns)}")
print(f"\nLeaf field count: {len(struct_sample_df.columns)}")
assert len(struct_sample_df.columns) == 31, (
    f"Expected 31 struct leaf fields, got {len(struct_sample_df.columns)}"
)

Struct leaf sample shape: (5, 31)
Columns extracted: ['details_gameSpeed', 'details_isBlizzardMap', 'details_timeUTC', 'header_elapsedGameLoops', 'header_version', 'go_advancedSharedControl', 'go_amm', 'go_battleNet', 'go_clientDebugFlags', 'go_competitive', 'go_cooperative', 'go_fog', 'go_heroDuplicatesAllowed', 'go_lockTeams', 'go_noVictoryOrDefeat', 'go_observers', 'go_practice', 'go_randomRaces', 'go_teamsTogether', 'go_userDifficulty', 'gd_gameSpeed', 'gd_isBlizzardMap', 'gd_mapAuthorName', 'gd_mapFileSyncChecksum', 'gd_mapSizeX', 'gd_mapSizeY', 'gd_maxPlayers', 'metadata_baseBuild', 'metadata_dataBuild', 'metadata_gameVersion', 'metadata_mapName']

Leaf field count: 31


## Cell 6 -- T01: Struct leaf field cardinality and null census (all 31 fields)

In [6]:
sql_queries["struct_leaf_null_cardinality"] = """
SELECT
    -- details.gameSpeed
    COUNT(*) AS n_rows,
    COUNT(DISTINCT details.gameSpeed) AS details_gameSpeed_card,
    SUM(CASE WHEN details.gameSpeed IS NULL THEN 1 ELSE 0 END)
        AS details_gameSpeed_nulls,

    -- details.isBlizzardMap
    COUNT(DISTINCT details.isBlizzardMap) AS details_isBlizzardMap_card,
    SUM(CASE WHEN details.isBlizzardMap IS NULL THEN 1 ELSE 0 END)
        AS details_isBlizzardMap_nulls,

    -- details.timeUTC
    COUNT(DISTINCT details.timeUTC) AS details_timeUTC_card,
    SUM(CASE WHEN details.timeUTC IS NULL THEN 1 ELSE 0 END)
        AS details_timeUTC_nulls,

    -- header.elapsedGameLoops
    COUNT(DISTINCT header.elapsedGameLoops) AS header_elapsed_card,
    SUM(CASE WHEN header.elapsedGameLoops IS NULL THEN 1 ELSE 0 END)
        AS header_elapsed_nulls,
    MIN(header.elapsedGameLoops) AS header_elapsed_min,
    MAX(header.elapsedGameLoops) AS header_elapsed_max,

    -- header.version
    COUNT(DISTINCT header.version) AS header_version_card,
    SUM(CASE WHEN header.version IS NULL THEN 1 ELSE 0 END)
        AS header_version_nulls,

    -- gameOptions fields
    COUNT(DISTINCT initData.gameDescription.gameOptions.advancedSharedControl)
        AS go_advancedSharedControl_card,
    COUNT(DISTINCT initData.gameDescription.gameOptions.amm)
        AS go_amm_card,
    COUNT(DISTINCT initData.gameDescription.gameOptions.battleNet)
        AS go_battleNet_card,
    COUNT(DISTINCT initData.gameDescription.gameOptions.clientDebugFlags)
        AS go_clientDebugFlags_card,
    COUNT(DISTINCT initData.gameDescription.gameOptions.competitive)
        AS go_competitive_card,
    COUNT(DISTINCT initData.gameDescription.gameOptions.cooperative)
        AS go_cooperative_card,
    COUNT(DISTINCT initData.gameDescription.gameOptions.fog)
        AS go_fog_card,
    COUNT(DISTINCT initData.gameDescription.gameOptions.heroDuplicatesAllowed)
        AS go_heroDuplicatesAllowed_card,
    COUNT(DISTINCT initData.gameDescription.gameOptions.lockTeams)
        AS go_lockTeams_card,
    COUNT(DISTINCT initData.gameDescription.gameOptions.noVictoryOrDefeat)
        AS go_noVictoryOrDefeat_card,
    COUNT(DISTINCT initData.gameDescription.gameOptions.observers)
        AS go_observers_card,
    COUNT(DISTINCT initData.gameDescription.gameOptions.practice)
        AS go_practice_card,
    COUNT(DISTINCT initData.gameDescription.gameOptions.randomRaces)
        AS go_randomRaces_card,
    COUNT(DISTINCT initData.gameDescription.gameOptions.teamsTogether)
        AS go_teamsTogether_card,
    COUNT(DISTINCT initData.gameDescription.gameOptions.userDifficulty)
        AS go_userDifficulty_card,

    -- initData.gameDescription direct fields
    COUNT(DISTINCT initData.gameDescription.gameSpeed)
        AS gd_gameSpeed_card,
    COUNT(DISTINCT initData.gameDescription.isBlizzardMap)
        AS gd_isBlizzardMap_card,
    COUNT(DISTINCT initData.gameDescription.mapAuthorName)
        AS gd_mapAuthorName_card,
    SUM(CASE WHEN initData.gameDescription.mapAuthorName IS NULL THEN 1 ELSE 0 END)
        AS gd_mapAuthorName_nulls,
    COUNT(DISTINCT initData.gameDescription.mapFileSyncChecksum)
        AS gd_mapFileSyncChecksum_card,
    COUNT(DISTINCT initData.gameDescription.mapSizeX) AS gd_mapSizeX_card,
    MIN(initData.gameDescription.mapSizeX) AS gd_mapSizeX_min,
    MAX(initData.gameDescription.mapSizeX) AS gd_mapSizeX_max,
    COUNT(DISTINCT initData.gameDescription.mapSizeY) AS gd_mapSizeY_card,
    MIN(initData.gameDescription.mapSizeY) AS gd_mapSizeY_min,
    MAX(initData.gameDescription.mapSizeY) AS gd_mapSizeY_max,
    COUNT(DISTINCT initData.gameDescription.maxPlayers) AS gd_maxPlayers_card,

    -- metadata fields
    COUNT(DISTINCT metadata.baseBuild)   AS metadata_baseBuild_card,
    COUNT(DISTINCT metadata.dataBuild)   AS metadata_dataBuild_card,
    COUNT(DISTINCT metadata.gameVersion) AS metadata_gameVersion_card,
    COUNT(DISTINCT metadata.mapName)     AS metadata_mapName_card,
    SUM(CASE WHEN metadata.mapName IS NULL THEN 1 ELSE 0 END)
        AS metadata_mapName_nulls

FROM replays_meta_raw
"""

struct_stats_df = conn.con.execute(sql_queries["struct_leaf_null_cardinality"]).df()
print("=== Struct leaf field statistics ===")
# Transpose for readability
print(struct_stats_df.T.to_string())

=== Struct leaf field statistics ===
                                      0
n_rows                          22390.0
details_gameSpeed_card              1.0
details_gameSpeed_nulls             0.0
details_isBlizzardMap_card          2.0
details_isBlizzardMap_nulls         0.0
details_timeUTC_card            22344.0
details_timeUTC_nulls               0.0
header_elapsed_card             14045.0
header_elapsed_nulls                0.0
header_elapsed_min                 12.0
header_elapsed_max             136028.0
header_version_card                46.0
header_version_nulls                0.0
go_advancedSharedControl_card       1.0
go_amm_card                         2.0
go_battleNet_card                   1.0
go_clientDebugFlags_card            2.0
go_competitive_card                 2.0
go_cooperative_card                 1.0
go_fog_card                         1.0
go_heroDuplicatesAllowed_card       1.0
go_lockTeams_card                   1.0
go_noVictoryOrDefeat_card           1.0
go_

## Cell 7 -- T01: Identify constant and near-constant gameOptions flags

In [7]:
sql_queries["go_flag_distributions"] = """
SELECT
    initData.gameDescription.gameOptions.advancedSharedControl AS advancedSharedControl,
    initData.gameDescription.gameOptions.amm               AS amm,
    initData.gameDescription.gameOptions.battleNet         AS battleNet,
    initData.gameDescription.gameOptions.competitive       AS competitive,
    initData.gameDescription.gameOptions.cooperative       AS cooperative,
    initData.gameDescription.gameOptions.fog               AS fog,
    initData.gameDescription.gameOptions.heroDuplicatesAllowed AS heroDuplicatesAllowed,
    initData.gameDescription.gameOptions.lockTeams         AS lockTeams,
    initData.gameDescription.gameOptions.noVictoryOrDefeat AS noVictoryOrDefeat,
    initData.gameDescription.gameOptions.practice          AS practice,
    initData.gameDescription.gameOptions.randomRaces       AS randomRaces,
    initData.gameDescription.gameOptions.teamsTogether     AS teamsTogether,
    COUNT(*) AS cnt
FROM replays_meta_raw
GROUP BY
    advancedSharedControl, amm, battleNet, competitive, cooperative,
    fog, heroDuplicatesAllowed, lockTeams, noVictoryOrDefeat,
    practice, randomRaces, teamsTogether
ORDER BY cnt DESC
LIMIT 20
"""

go_flags_df = conn.con.execute(sql_queries["go_flag_distributions"]).df()
print("=== gameOptions flag combinations (top 20 by frequency) ===")
print(go_flags_df.to_string(index=False))

=== gameOptions flag combinations (top 20 by frequency) ===
 advancedSharedControl   amm  battleNet  competitive  cooperative  fog  heroDuplicatesAllowed  lockTeams  noVictoryOrDefeat  practice  randomRaces  teamsTogether   cnt
                 False False       True        False        False    0                   True       True              False     False        False          False 22387
                 False  True       True         True        False    0                   True       True              False     False        False          False     3


## Cell 8 -- T01: Join verification using replay_id (I6, sql-data.md rule)

Per sql-data.md: join key must be replay_id derived via
regexp_extract(filename, '([0-9a-f]{32})\.SC2Replay\.json$', 1).
NEVER join directly on raw filename.

In [8]:
sql_queries["join_verification_replay_id"] = """
WITH rp_distinct AS (
    SELECT DISTINCT
        regexp_extract(filename, '([0-9a-f]{32})\\.SC2Replay\\.json$', 1) AS replay_id
    FROM replay_players_raw
    WHERE regexp_extract(filename, '([0-9a-f]{32})\\.SC2Replay\\.json$', 1) != ''
),
rm_distinct AS (
    SELECT DISTINCT
        regexp_extract(filename, '([0-9a-f]{32})\\.SC2Replay\\.json$', 1) AS replay_id
    FROM replays_meta_raw
    WHERE regexp_extract(filename, '([0-9a-f]{32})\\.SC2Replay\\.json$', 1) != ''
),
matched AS (
    SELECT rp.replay_id
    FROM rp_distinct rp
    JOIN rm_distinct rm ON rp.replay_id = rm.replay_id
),
rp_only AS (
    SELECT rp.replay_id
    FROM rp_distinct rp
    LEFT JOIN rm_distinct rm ON rp.replay_id = rm.replay_id
    WHERE rm.replay_id IS NULL
),
rm_only AS (
    SELECT rm.replay_id
    FROM rm_distinct rm
    LEFT JOIN rp_distinct rp ON rm.replay_id = rp.replay_id
    WHERE rp.replay_id IS NULL
)
SELECT
    (SELECT COUNT(*) FROM rp_distinct) AS rp_distinct_replay_ids,
    (SELECT COUNT(*) FROM rm_distinct) AS rm_distinct_replay_ids,
    (SELECT COUNT(*) FROM matched)     AS matched_replay_ids,
    (SELECT COUNT(*) FROM rp_only)     AS rp_only,
    (SELECT COUNT(*) FROM rm_only)     AS rm_only
"""

join_df = conn.con.execute(sql_queries["join_verification_replay_id"]).df()
print("=== Join verification via replay_id ===")
print(join_df.to_string(index=False))

rp_ids_count = int(join_df["rp_distinct_replay_ids"].iloc[0])
rm_ids_count = int(join_df["rm_distinct_replay_ids"].iloc[0])
matched = int(join_df["matched_replay_ids"].iloc[0])
rp_only = int(join_df["rp_only"].iloc[0])
rm_only = int(join_df["rm_only"].iloc[0])

print(f"\nrp distinct replay_ids : {rp_ids_count}")
print(f"rm distinct replay_ids : {rm_ids_count}")
print(f"Matched (intersection) : {matched}")
print(f"rp only (orphans)      : {rp_only}")
print(f"rm only (orphans)      : {rm_only}")
print(f"Match rate             : {matched / rm_ids_count * 100:.4f}%")

=== Join verification via replay_id ===
 rp_distinct_replay_ids  rm_distinct_replay_ids  matched_replay_ids  rp_only  rm_only
                  22390                   22390               22390        0        0

rp distinct replay_ids : 22390
rm distinct replay_ids : 22390
Matched (intersection) : 22390
rp only (orphans)      : 0
rm only (orphans)      : 0
Match rate             : 100.0000%


## Cell 9 -- T01: Verify regex extraction works on sample filenames

In [9]:
sql_queries["replay_id_sample"] = """
SELECT
    filename,
    regexp_extract(filename, '([0-9a-f]{32})\\.SC2Replay\\.json$', 1) AS replay_id
FROM replays_meta_raw
LIMIT 10
"""

replay_id_sample_df = conn.con.execute(sql_queries["replay_id_sample"]).df()
print("=== Sample replay_id extraction ===")
print(replay_id_sample_df.to_string(index=False))

# Verify no empty replay_ids in replays_meta_raw
sql_queries["empty_replay_id_check"] = """
SELECT COUNT(*) AS empty_count
FROM replays_meta_raw
WHERE regexp_extract(filename, '([0-9a-f]{32})\\.SC2Replay\\.json$', 1) = ''
"""

empty_df = conn.con.execute(sql_queries["empty_replay_id_check"]).df()
n_empty = int(empty_df["empty_count"].iloc[0])
print(f"\nReplays_meta_raw rows with empty replay_id: {n_empty}")
assert n_empty == 0, f"Unexpected empty replay_ids in replays_meta_raw: {n_empty}"

=== Sample replay_id extraction ===
                                                                                  filename                        replay_id
2016_IEM_10_Taipei/2016_IEM_10_Taipei_data/095724b86cbca0e6da2fb8baad0d7baf.SC2Replay.json 095724b86cbca0e6da2fb8baad0d7baf
2016_IEM_10_Taipei/2016_IEM_10_Taipei_data/0e0b1a550447f0b0a616e48224b31bd9.SC2Replay.json 0e0b1a550447f0b0a616e48224b31bd9
2016_IEM_10_Taipei/2016_IEM_10_Taipei_data/15ad08e3ed7fc167bf455dbe3f1f7d91.SC2Replay.json 15ad08e3ed7fc167bf455dbe3f1f7d91
2016_IEM_10_Taipei/2016_IEM_10_Taipei_data/1a9aeda55b1c48e503b94aca122f41a5.SC2Replay.json 1a9aeda55b1c48e503b94aca122f41a5
2016_IEM_10_Taipei/2016_IEM_10_Taipei_data/278b7a0a0b2b8ca4594d45e5b0b0ceec.SC2Replay.json 278b7a0a0b2b8ca4594d45e5b0b0ceec
2016_IEM_10_Taipei/2016_IEM_10_Taipei_data/3813bc1121bf0b258ea224918ff6f6f7.SC2Replay.json 3813bc1121bf0b258ea224918ff6f6f7
2016_IEM_10_Taipei/2016_IEM_10_Taipei_data/407c8d1441b9d43ae3211b3b5b9664af.SC2Replay.json 407c8

## Cell 10 -- T02: map_aliases_raw utility assessment

Question: Are map names in replays_meta_raw (metadata.mapName) already in
English? If yes, map_aliases_raw (104K rows, 70 identical copies of 1,488
key-value pairs) may be unnecessary.

In [10]:
sql_queries["meta_map_name_sample"] = """
SELECT
    metadata.mapName AS map_name,
    COUNT(*) AS replay_count,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 4) AS pct
FROM replays_meta_raw
GROUP BY map_name
ORDER BY replay_count DESC
LIMIT 30
"""

map_name_df = conn.con.execute(sql_queries["meta_map_name_sample"]).df()
print("=== Top 30 map names in replays_meta_raw.metadata.mapName ===")
print(map_name_df.to_string(index=False))

=== Top 30 map names in replays_meta_raw.metadata.mapName ===
             map_name  replay_count    pct
  2000 Atmospheres LE           966 4.3144
        Acid Plant LE           667 2.9790
          Catalyst LE           586 2.6172
             Oxide LE           570 2.5458
       Romanticide LE           568 2.5368
    Eternal Empire LE           528 2.3582
       King's Cove LE           509 2.2733
        Lightshade LE           508 2.2689
   Kairos Junction LE           496 2.2153
      Abyssal Reef LE           481 2.1483
        Ever Dream LE           471 2.1036
    Lost and Found LE           461 2.0590
        Jagannatha LE           440 1.9652
         Deathaura LE           438 1.9562
         Acropolis LE           419 1.8714
   Pillars of Gold LE           390 1.7418
         [ESL] Data-C           387 1.7285
   Port Aleksander LE           381 1.7017
         Blackpink LE           370 1.6525
      Cyber Forest LE           364 1.6257
       Thunderbird LE           362

## Cell 11 -- T02: Check map_aliases_raw for non-ASCII foreign names

In [11]:
sql_queries["map_aliases_raw_overview"] = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT tournament) AS distinct_tournaments,
    COUNT(DISTINCT foreign_name) AS distinct_foreign_names,
    COUNT(DISTINCT english_name) AS distinct_english_names,
    COUNT(DISTINCT filename) AS distinct_files
FROM map_aliases_raw
"""

aliases_overview_df = conn.con.execute(sql_queries["map_aliases_raw_overview"]).df()
print("=== map_aliases_raw overview ===")
print(aliases_overview_df.to_string(index=False))

=== map_aliases_raw overview ===
 total_rows  distinct_tournaments  distinct_foreign_names  distinct_english_names  distinct_files
     104160                    70                    1488                     215              70


## Cell 12 -- T02: Sample foreign names to determine if non-ASCII characters present

In [12]:
sql_queries["map_aliases_sample"] = """
SELECT foreign_name, english_name
FROM map_aliases_raw
WHERE tournament = (SELECT MIN(tournament) FROM map_aliases_raw)
ORDER BY foreign_name
LIMIT 30
"""

aliases_sample_df = conn.con.execute(sql_queries["map_aliases_sample"]).df()
print("=== Sample from map_aliases_raw (one tournament) ===")
print(aliases_sample_df.to_string(index=False))

=== Sample from map_aliases_raw (one tournament) ===
       foreign_name        english_name
         16 Bits LE           16-Bit LE
         16 bits EC           16-Bit LE
         16 bits EE           16-Bit LE
         16 bits EJ           16-Bit LE
        16 bitów ER           16-Bit LE
          16 бит РВ           16-Bit LE
          16-Bit LE           16-Bit LE
            16位-天梯版           16-Bit LE
         16位元 - 天梯版           16-Bit LE
          16비트 - 래더           16-Bit LE
2.000 Atmosferas LE 2000 Atmospheres LE
 2.000 Atmosfere LE 2000 Atmospheres LE
2000 Atmospheres LE 2000 Atmospheres LE
2000 Atmosphären LE 2000 Atmospheres LE
 2000 Atmósferas EE 2000 Atmospheres LE
   2000 atmosfer ER 2000 Atmospheres LE
 2000 atmósferas EJ 2000 Atmospheres LE
   2000 атмосфер РВ 2000 Atmospheres LE
   2000 애트모스피어 - 래더 2000 Atmospheres LE
     2000大氣壓力 - 天梯版 2000 Atmospheres LE
      Abiogenese LE      Abiogenesis LE
      Abiogenesi LE      Abiogenesis LE
     Abiogenesis LE      Ab

## Cell 13 -- T02: Cross-reference: do any meta map names appear in foreign_name column?

In [13]:
sql_queries["map_name_vs_aliases_cross"] = """
WITH meta_maps AS (
    SELECT DISTINCT metadata.mapName AS map_name
    FROM replays_meta_raw
    WHERE metadata.mapName IS NOT NULL
)
SELECT
    COUNT(*) AS meta_map_count,
    COUNT(*) FILTER (
        WHERE map_name IN (SELECT DISTINCT foreign_name FROM map_aliases_raw)
    ) AS in_foreign_names,
    COUNT(*) FILTER (
        WHERE map_name IN (SELECT DISTINCT english_name FROM map_aliases_raw)
    ) AS in_english_names,
    COUNT(*) FILTER (
        WHERE map_name NOT IN (SELECT DISTINCT foreign_name FROM map_aliases_raw)
        AND map_name NOT IN (SELECT DISTINCT english_name FROM map_aliases_raw)
    ) AS in_neither
FROM meta_maps
"""

cross_df = conn.con.execute(sql_queries["map_name_vs_aliases_cross"]).df()
print("=== Cross-reference: meta map names vs alias table ===")
print(cross_df.to_string(index=False))
print("\nInterpretation:")
print("  in_foreign_names > 0 => some map names are NOT yet translated")
print("  in_english_names = meta_map_count => all meta names are already English")
print("  in_neither > 0 => names not in alias table at all (custom/unknown maps)")

=== Cross-reference: meta map names vs alias table ===
 meta_map_count  in_foreign_names  in_english_names  in_neither
            188               188               188           0

Interpretation:
  in_foreign_names > 0 => some map names are NOT yet translated
  in_english_names = meta_map_count => all meta names are already English
  in_neither > 0 => names not in alias table at all (custom/unknown maps)


## Cell 14 -- T02: Check if foreign_name == english_name for any entries
(determines if the translation is always meaningful)

In [14]:
sql_queries["alias_identity_check"] = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE foreign_name = english_name) AS identity_count,
    COUNT(*) FILTER (WHERE foreign_name != english_name) AS translated_count,
    ROUND(
        100.0 * COUNT(*) FILTER (WHERE foreign_name = english_name) / COUNT(*),
        2
    ) AS identity_pct
FROM (SELECT DISTINCT foreign_name, english_name FROM map_aliases_raw)
"""

alias_identity_df = conn.con.execute(sql_queries["alias_identity_check"]).df()
print("=== map_aliases_raw: identity vs translation ===")
print(alias_identity_df.to_string(index=False))

=== map_aliases_raw: identity vs translation ===
 total_rows  identity_count  translated_count  identity_pct
       1489             215              1274         14.44


## Cell 15 -- T02: Map name null check in replays_meta_raw

In [15]:
sql_queries["map_name_null_check"] = """
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN metadata.mapName IS NULL THEN 1 ELSE 0 END) AS null_count,
    COUNT(DISTINCT metadata.mapName) AS distinct_map_names
FROM replays_meta_raw
"""

map_null_df = conn.con.execute(sql_queries["map_name_null_check"]).df()
print("=== metadata.mapName null analysis ===")
print(map_null_df.to_string(index=False))

=== metadata.mapName null analysis ===
 total_rows  null_count  distinct_map_names
      22390         0.0                 188


## Cell 16 -- T03: Event view temporal characterization (game_events_raw)

DO NOT run COUNT(*) on game_events_raw (608M rows -- will timeout).
Row count sourced from schema YAML (608,618,823).
Query evtTypeName at loop=0 to characterize initialization events.
Use LIMIT to avoid full scan.

In [16]:
GAME_EVENTS_ROW_COUNT = 608_618_823  # from game_events_raw.yaml (I7: no magic numbers)
TRACKER_EVENTS_ROW_COUNT = 62_003_411  # from tracker_events_raw.yaml
MESSAGE_EVENTS_ROW_COUNT = 52_167  # from message_events_raw.yaml

print(f"game_events_raw rows (schema YAML):    {GAME_EVENTS_ROW_COUNT:,}")
print(f"tracker_events_raw rows (schema YAML): {TRACKER_EVENTS_ROW_COUNT:,}")
print(f"message_events_raw rows (schema YAML): {MESSAGE_EVENTS_ROW_COUNT:,}")

game_events_raw rows (schema YAML):    608,618,823
tracker_events_raw rows (schema YAML): 62,003,411
message_events_raw rows (schema YAML): 52,167


## Cell 17 -- T03: Characterize loop=0 events in game_events_raw

In [17]:
sql_queries["game_events_loop0_types"] = """
SELECT
    evtTypeName,
    COUNT(*) AS cnt
FROM game_events_raw
WHERE loop = 0
GROUP BY evtTypeName
ORDER BY cnt DESC
"""

# Use a timeout-safe approach: sample a batch file to characterize
# loop=0 events rather than full-scan. We inspect the first batch.
game_loop0_df = conn.con.execute(sql_queries["game_events_loop0_types"]).df()
print("=== game_events_raw: evtTypeName distribution at loop=0 ===")
print(game_loop0_df.to_string(index=False))
print(f"\nTotal loop=0 event types: {len(game_loop0_df)}")

=== game_events_raw: evtTypeName distribution at loop=0 ===
               evtTypeName    cnt
               UserOptions 131738
DecrementGameTimeRemaining   1006
                CameraSave    533
             GameUserLeave     51
      TriggerDialogControl     18

Total loop=0 event types: 5


## Cell 18 -- T03: Min loop per replay in game_events_raw (sample-based)

To check whether pre-game events (loop < 0) exist, we sample
the minimum loop value per replay from one batch file.

In [18]:
sql_queries["game_events_min_loop_sample"] = """
SELECT
    MIN(loop) AS global_min_loop,
    MAX(loop) AS global_max_loop,
    COUNT(DISTINCT evtTypeName) AS distinct_event_types
FROM game_events_raw
WHERE loop <= 10
"""

# This is bounded to loop<=10 so it will scan only early events
game_min_loop_df = conn.con.execute(
    sql_queries["game_events_min_loop_sample"]
).df()
print("=== game_events_raw: loop range check (loop <= 10) ===")
print(game_min_loop_df.to_string(index=False))

=== game_events_raw: loop range check (loop <= 10) ===
 global_min_loop  global_max_loop  distinct_event_types
               0               10                    13


## Cell 19 -- T03: Event type distribution at loop=0 in tracker_events_raw
(62M rows -- feasible to count)

In [19]:
sql_queries["tracker_events_loop0_types"] = """
SELECT
    evtTypeName,
    COUNT(*) AS cnt
FROM tracker_events_raw
WHERE loop = 0
GROUP BY evtTypeName
ORDER BY cnt DESC
"""

tracker_loop0_df = conn.con.execute(sql_queries["tracker_events_loop0_types"]).df()
print("=== tracker_events_raw: evtTypeName distribution at loop=0 ===")
print(tracker_loop0_df.to_string(index=False))
print(f"\nTotal loop=0 event types in tracker_events_raw: {len(tracker_loop0_df)}")

=== tracker_events_raw: evtTypeName distribution at loop=0 ===
evtTypeName     cnt
   UnitBorn 4962996
    Upgrade  263068
PlayerSetup   44817
PlayerStats       1

Total loop=0 event types in tracker_events_raw: 4


## Cell 20 -- T04: tracker_events_raw full evtTypeName distribution
(62M rows -- feasible COUNT)

In [20]:
sql_queries["tracker_events_type_distribution"] = """
SELECT
    evtTypeName,
    COUNT(*) AS cnt,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 4) AS pct
FROM tracker_events_raw
GROUP BY evtTypeName
ORDER BY cnt DESC
"""

tracker_type_df = conn.con.execute(
    sql_queries["tracker_events_type_distribution"]
).df()
print("=== tracker_events_raw: evtTypeName distribution (full scan) ===")
print(tracker_type_df.to_string(index=False))
print(f"\nTotal rows (from COUNT): {tracker_type_df['cnt'].sum():,}")
print(f"Expected (schema YAML):  {TRACKER_EVENTS_ROW_COUNT:,}")

=== tracker_events_raw: evtTypeName distribution (full scan) ===
    evtTypeName      cnt     pct
       UnitBorn 22372489 36.0827
       UnitDied 16053834 25.8919
 UnitTypeChange 10999108 17.7395
    PlayerStats  4558736  7.3524
       UnitInit  3151270  5.0824
       UnitDone  3018764  4.8687
  UnitPositions   941249  1.5181
        Upgrade   797987  1.2870
UnitOwnerChange    65157  0.1051
    PlayerSetup    44817  0.0723

Total rows (from COUNT): 62,003,411
Expected (schema YAML):  62,003,411


## Cell 21 -- T04: tracker_events_raw min/max loop

In [21]:
sql_queries["tracker_events_loop_range"] = """
SELECT
    MIN(loop) AS min_loop,
    MAX(loop) AS max_loop,
    COUNT(DISTINCT filename) AS distinct_replays
FROM tracker_events_raw
"""

tracker_range_df = conn.con.execute(
    sql_queries["tracker_events_loop_range"]
).df()
print("=== tracker_events_raw: loop range and replay coverage ===")
print(tracker_range_df.to_string(index=False))

=== tracker_events_raw: loop range and replay coverage ===
 min_loop  max_loop  distinct_replays
        0    109287             22390


## Cell 22 -- T05: message_events_raw utility assessment
(52K rows -- full scan feasible)

In [22]:
sql_queries["message_events_overview"] = """
SELECT
    evtTypeName,
    COUNT(*) AS cnt,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 4) AS pct
FROM message_events_raw
GROUP BY evtTypeName
ORDER BY cnt DESC
"""

msg_type_df = conn.con.execute(sql_queries["message_events_overview"]).df()
print("=== message_events_raw: evtTypeName distribution ===")
print(msg_type_df.to_string(index=False))
print(f"\nTotal rows: {msg_type_df['cnt'].sum():,}")

=== message_events_raw: evtTypeName distribution ===
    evtTypeName   cnt     pct
           Chat 51412 98.5527
           Ping   714  1.3687
ReconnectNotify    41  0.0786

Total rows: 52,167


## Cell 23 -- T05: message_events_raw replay coverage

In [23]:
sql_queries["message_events_coverage"] = """
SELECT
    COUNT(DISTINCT filename) AS distinct_replays_in_events,
    COUNT(*) AS total_rows,
    MIN(loop) AS min_loop,
    MAX(loop) AS max_loop
FROM message_events_raw
"""

msg_coverage_df = conn.con.execute(sql_queries["message_events_coverage"]).df()
print("=== message_events_raw: coverage ===")
print(msg_coverage_df.to_string(index=False))
print(f"\nOut of {RM_TOTAL_ROWS} total replays in replays_meta_raw:")
msg_replays = int(msg_coverage_df["distinct_replays_in_events"].iloc[0])
print(
    f"  Replays with message events: {msg_replays} "
    f"({100.0 * msg_replays / RM_TOTAL_ROWS:.2f}%)"
)
print(
    f"  Replays with NO message events: {RM_TOTAL_ROWS - msg_replays} "
    f"({100.0 * (RM_TOTAL_ROWS - msg_replays) / RM_TOTAL_ROWS:.2f}%)"
)

=== message_events_raw: coverage ===
 distinct_replays_in_events  total_rows  min_loop  max_loop
                      22260       52167         0    109280

Out of 22390 total replays in replays_meta_raw:
  Replays with message events: 22260 (99.42%)
  Replays with NO message events: 130 (0.58%)


## Cell 24 -- T05: Sample message event content

In [24]:
sql_queries["message_events_sample"] = """
SELECT
    evtTypeName,
    loop,
    event_data
FROM message_events_raw
ORDER BY loop
LIMIT 10
"""

msg_sample_df = conn.con.execute(sql_queries["message_events_sample"]).df()
print("=== message_events_raw: sample rows ===")
print(msg_sample_df.to_string(index=False))

=== message_events_raw: sample rows ===
    evtTypeName  loop                                                                                              event_data
ReconnectNotify     0            {"evtTypeName": "ReconnectNotify", "id": 4, "loop": 0, "status": 2, "userid": {"userId": 2}}
ReconnectNotify     0            {"evtTypeName": "ReconnectNotify", "id": 4, "loop": 0, "status": 1, "userid": {"userId": 2}}
           Chat     0 {"evtTypeName": "Chat", "id": 0, "loop": 0, "recipient": 0, "string": "gl hf", "userid": {"userId": 1}}
           Chat     0 {"evtTypeName": "Chat", "id": 0, "loop": 0, "recipient": 0, "string": "gl hf", "userid": {"userId": 2}}
           Chat     0  {"evtTypeName": "Chat", "id": 0, "loop": 0, "recipient": 0, "string": "okok", "userid": {"userId": 1}}
           Chat     0 {"evtTypeName": "Chat", "id": 0, "loop": 0, "recipient": 0, "string": "gl hf", "userid": {"userId": 3}}
           Chat     0    {"evtTypeName": "Chat", "id": 0, "loop": 0, "recipien

## Cell 25 -- T06: replays_meta_raw additional utility fields
(gameVersion, elapsed_game_loops, timestamp distributions)

In [25]:
sql_queries["meta_version_distribution"] = """
SELECT
    metadata.gameVersion AS game_version,
    COUNT(*) AS replay_count,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 4) AS pct
FROM replays_meta_raw
GROUP BY game_version
ORDER BY replay_count DESC
LIMIT 20
"""

version_df = conn.con.execute(sql_queries["meta_version_distribution"]).df()
print("=== Game version distribution (top 20) ===")
print(version_df.to_string(index=False))
print(f"\nDistinct game versions: {len(version_df)}")

=== Game version distribution (top 20) ===
game_version  replay_count     pct
 5.0.7.84643          2511 11.2148
 5.0.9.87702          1094  4.8861
 5.0.8.86383          1071  4.7834
5.0.10.88500           988  4.4127
 4.8.3.72282           933  4.1670
5.0.10.89165           865  3.8633
5.0.12.91115           859  3.8365
5.0.11.90136           820  3.6623
 5.0.2.81433           799  3.5686
4.11.4.78285           795  3.5507
4.12.0.80188           783  3.4971
 4.9.3.75025           765  3.4167
 4.4.0.65895           753  3.3631
4.10.2.76052           601  2.6842
 4.8.6.73620           540  2.4118
 4.1.4.61545           482  2.1527
 5.0.6.83830           444  1.9830
 4.2.0.62347           444  1.9830
 4.6.0.67926           442  1.9741
5.0.13.92174           439  1.9607

Distinct game versions: 20


## Cell 26 -- T06: timestamp distribution (details.timeUTC)

In [26]:
sql_queries["timestamp_distribution"] = """
SELECT
    EXTRACT(YEAR FROM TRY_CAST(details.timeUTC AS TIMESTAMP)) AS year,
    COUNT(*) AS replay_count,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 4) AS pct,
    MIN(details.timeUTC) AS sample_min,
    MAX(details.timeUTC) AS sample_max
FROM replays_meta_raw
GROUP BY year
ORDER BY year
"""

timestamp_df = conn.con.execute(sql_queries["timestamp_distribution"]).df()
print("=== Replay timestamp distribution by year (details.timeUTC) ===")
print(timestamp_df.to_string(index=False))

=== Replay timestamp distribution by year (details.timeUTC) ===
 year  replay_count     pct                   sample_min                   sample_max
 2016           555  2.4788     2016-01-07T02:21:46.002Z 2016-07-30T03:56:37.0212208Z
 2017          1999  8.9281 2017-02-27T08:47:32.1767923Z 2017-11-26T16:49:12.5469012Z
 2018          3180 14.2028 2018-01-26T11:16:14.8721833Z 2018-12-10T06:13:08.9725988Z
 2019          3886 17.3560 2019-01-31T23:16:51.6922468Z 2019-11-24T22:19:47.8864437Z
 2020          2842 12.6932  2020-02-24T09:11:18.735104Z 2020-12-21T05:05:59.6587006Z
 2021          3836 17.1326 2021-01-12T14:13:07.4144021Z 2021-12-20T08:12:21.9804449Z
 2022          3294 14.7119 2022-01-11T13:08:59.0895415Z 2022-12-18T21:55:53.2141047Z
 2023          1752  7.8249 2023-02-08T10:13:08.3801193Z 2023-12-18T06:13:06.2466371Z
 2024          1046  4.6717 2024-02-08T10:10:30.3179032Z 2024-12-01T23:48:45.2511615Z


## Cell 27 -- T06: Table utility verdict summary

In [27]:
# Build the verdict dict from empirical results gathered above
verdict = {
    "step": "01_03_03",
    "dataset": "sc2egset",
    "table_verdicts": {},
    "join_key": {
        "method": "replay_id via regexp_extract(filename, '([0-9a-f]{32})\\.SC2Replay\\.json$', 1)",
        "rp_distinct_replay_ids": int(join_df["rp_distinct_replay_ids"].iloc[0]),
        "rm_distinct_replay_ids": int(join_df["rm_distinct_replay_ids"].iloc[0]),
        "matched_replay_ids": int(join_df["matched_replay_ids"].iloc[0]),
        "rp_orphan_count": int(join_df["rp_only"].iloc[0]),
        "rm_orphan_count": int(join_df["rm_only"].iloc[0]),
    },
    "struct_leaf_fields": {
        "total_extracted": len(struct_sample_df.columns),
        "confirmed_31_fields": len(struct_sample_df.columns) == 31,
    },
    "map_name_analysis": {
        "meta_map_count": int(map_null_df["distinct_map_names"].iloc[0]),
        "meta_mapName_nulls": int(map_null_df["null_count"].iloc[0]),
        "in_foreign_names": int(cross_df["in_foreign_names"].iloc[0]),
        "in_english_names": int(cross_df["in_english_names"].iloc[0]),
        "in_neither": int(cross_df["in_neither"].iloc[0]),
        "alias_identity_pct": float(alias_identity_df["identity_pct"].iloc[0]),
    },
    "event_row_counts": {
        "game_events_raw": GAME_EVENTS_ROW_COUNT,
        "game_events_raw_source": "schema YAML (too large to COUNT)",
        "tracker_events_raw": int(tracker_type_df["cnt"].sum()),
        "tracker_events_raw_source": "live COUNT from DuckDB",
        "message_events_raw": int(msg_type_df["cnt"].sum()),
        "message_events_raw_source": "live COUNT from DuckDB",
    },
    "tracker_events_loop_range": {
        "min_loop": int(tracker_range_df["min_loop"].iloc[0]),
        "max_loop": int(tracker_range_df["max_loop"].iloc[0]),
        "distinct_replays": int(tracker_range_df["distinct_replays"].iloc[0]),
    },
    "message_events_coverage": {
        "distinct_replays": int(msg_coverage_df["distinct_replays_in_events"].iloc[0]),
        "total_replays": RM_TOTAL_ROWS,
        "coverage_pct": round(
            100.0 * int(msg_coverage_df["distinct_replays_in_events"].iloc[0])
            / RM_TOTAL_ROWS,
            4,
        ),
    },
    "sql_queries": sql_queries,
}

# Table utility verdicts (derived empirically from query results above)
in_foreign = int(cross_df["in_foreign_names"].iloc[0])
in_english = int(cross_df["in_english_names"].iloc[0])
meta_map_count = int(map_null_df["distinct_map_names"].iloc[0])
alias_identity_pct = float(alias_identity_df["identity_pct"].iloc[0])
tracker_loop_min = int(tracker_range_df["min_loop"].iloc[0])

verdict["table_verdicts"]["replay_players_raw"] = {
    "verdict": "ESSENTIAL",
    "reason": (
        "Contains per-player features (MMR, race, result, APM, SQ, "
        "highestLeague) and the prediction target (result). "
        "25 columns, 44,817 rows. No equivalent in other tables."
    ),
    "i3_classes": ["PRE_GAME features", "IN_GAME metrics", "TARGET (result)"],
}

verdict["table_verdicts"]["replays_meta_raw"] = {
    "verdict": "ESSENTIAL",
    "reason": (
        "Contains match-level metadata: timestamp (details.timeUTC), "
        "game version (metadata.gameVersion), map name (metadata.mapName), "
        "and 31 struct leaf fields. Required for temporal ordering (I3) "
        "and map feature derivation. Join to replay_players_raw via replay_id."
    ),
    "struct_leaf_fields": 31,
    "i3_classes": [
        "PRE_GAME (timestamp, game_version, map_name, map_size, maxPlayers)",
        "POST_GAME (header.elapsedGameLoops)",
        "CONSTANT (game_speed, gameOptions flags mostly constant)",
    ],
}

verdict["table_verdicts"]["map_aliases_raw"] = {
    "verdict": (
        "UTILITY_CONDITIONAL"
        if in_foreign > 0
        else "LOW_UTILITY"
    ),
    "reason": (
        f"metadata.mapName in replays_meta_raw: {meta_map_count} distinct names. "
        f"{in_english} of {meta_map_count} are in the english_name column of "
        f"map_aliases_raw (already English). "
        f"{in_foreign} of {meta_map_count} appear in foreign_name column. "
        f"Identity (foreign==english) in alias table: {alias_identity_pct:.1f}% of pairs. "
        "Verdict depends on query results above."
    ),
    "row_count": 104160,
    "distinct_mapping_pairs": int(alias_identity_df["total_rows"].iloc[0]),
    "i3_class": "PRE_GAME (map translation lookup, no temporal info)",
}

verdict["table_verdicts"]["game_events_raw"] = {
    "verdict": "IN_GAME_ONLY",
    "reason": (
        f"608,618,823 rows of in-game action events. "
        f"Min loop at loop=0 events includes initialization events "
        f"(PlayerSetup, UnitBorn) that are part of game initialization, not "
        "pre-game state. All events are at loop >= 0. "
        "Excluded from Phase 02 pre-game feature pipeline per I3. "
        "Retained as a potential source for in-game feature comparison studies."
    ),
    "row_count": GAME_EVENTS_ROW_COUNT,
    "i3_class": "IN_GAME (loop >= 0, game-time events only)",
    "scan_note": "Row count from schema YAML (full COUNT would timeout on 608M rows)",
}

verdict["table_verdicts"]["tracker_events_raw"] = {
    "verdict": "IN_GAME_ONLY",
    "reason": (
        f"62,003,411 rows of tracker events (PlayerSetup, UnitBorn, etc). "
        f"Min loop: {tracker_loop_min}. All events represent game-time state. "
        "Excluded from Phase 02 pre-game feature pipeline per I3. "
        "May be used for in-game state features in later phases."
    ),
    "row_count": int(tracker_type_df["cnt"].sum()),
    "min_loop": tracker_loop_min,
    "i3_class": "IN_GAME (loop >= 0, game initialization + game-time state)",
}

verdict["table_verdicts"]["message_events_raw"] = {
    "verdict": "LOW_UTILITY",
    "reason": (
        f"52,167 rows in {int(msg_coverage_df['distinct_replays_in_events'].iloc[0])} "
        f"of {RM_TOTAL_ROWS} replays "
        f"({100.0 * int(msg_coverage_df['distinct_replays_in_events'].iloc[0]) / RM_TOTAL_ROWS:.2f}% coverage). "
        "Contains chat/ping messages between players during game. "
        "Not suitable as pre-game features (I3 violation). "
        "Insufficient coverage and content value for in-game features."
    ),
    "row_count": int(msg_type_df["cnt"].sum()),
    "i3_class": "IN_GAME (chat/ping messages during match)",
}

print("=== Table utility verdicts ===")
for tbl, v in verdict["table_verdicts"].items():
    print(f"\n{tbl}: {v['verdict']}")
    print(f"  {v['reason'][:120]}...")

=== Table utility verdicts ===

replay_players_raw: ESSENTIAL
  Contains per-player features (MMR, race, result, APM, SQ, highestLeague) and the prediction target (result). 25 columns,...

replays_meta_raw: ESSENTIAL
  Contains match-level metadata: timestamp (details.timeUTC), game version (metadata.gameVersion), map name (metadata.mapN...

map_aliases_raw: UTILITY_CONDITIONAL
  metadata.mapName in replays_meta_raw: 188 distinct names. 188 of 188 are in the english_name column of map_aliases_raw (...

game_events_raw: IN_GAME_ONLY
  608,618,823 rows of in-game action events. Min loop at loop=0 events includes initialization events (PlayerSetup, UnitBo...

tracker_events_raw: IN_GAME_ONLY
  62,003,411 rows of tracker events (PlayerSetup, UnitBorn, etc). Min loop: 0. All events represent game-time state. Exclu...

message_events_raw: LOW_UTILITY
  52,167 rows in 22260 of 22390 replays (99.42% coverage). Contains chat/ping messages between players during game. Not su...


## Cell 28 -- Save artifact JSON

In [28]:
output_json = artifact_dir / "01_03_03_table_utility.json"
with open(output_json, "w") as f:
    json.dump(verdict, f, indent=2, default=str)
print(f"Saved: {output_json}")

Saved: /Users/tomaszpionka/Projects/rts-outcome-prediction/src/rts_predict/games/sc2/datasets/sc2egset/reports/artifacts/01_exploration/03_profiling/01_03_03_table_utility.json


## Cell 29 -- Generate Markdown report

In [29]:
def _verdict_badge(v: str) -> str:
    mapping = {
        "ESSENTIAL": "**ESSENTIAL**",
        "IN_GAME_ONLY": "_IN_GAME_ONLY_",
        "LOW_UTILITY": "_LOW_UTILITY_",
        "UTILITY_CONDITIONAL": "_UTILITY_CONDITIONAL_",
    }
    return mapping.get(v, v)


md_lines = [
    "# Step 01_03_03 -- Table Utility Assessment",
    "## sc2egset Dataset",
    "",
    "**Phase:** 01 -- Data Exploration",
    "**Pipeline Section:** 01_03 -- Systematic Data Profiling",
    "**Predecessor:** 01_03_02",
    "**Date:** 2026-04-16",
    "**Invariants applied:** #3, #6, #9",
    "",
    "---",
    "",
    "## Join Key Verification",
    "",
    "Join key: `replay_id` derived via:",
    "```sql",
    "regexp_extract(filename, '([0-9a-f]{32})\\.SC2Replay\\.json$', 1) AS replay_id",
    "```",
    "",
    f"| Metric | Value |",
    f"|--------|-------|",
    f"| rp distinct replay_ids | {verdict['join_key']['rp_distinct_replay_ids']:,} |",
    f"| rm distinct replay_ids | {verdict['join_key']['rm_distinct_replay_ids']:,} |",
    f"| Matched (intersection) | {verdict['join_key']['matched_replay_ids']:,} |",
    f"| rp orphans | {verdict['join_key']['rp_orphan_count']} |",
    f"| rm orphans | {verdict['join_key']['rm_orphan_count']} |",
    "",
    "---",
    "",
    "## Struct Leaf Field Enumeration",
    "",
    (
        f"replays_meta_raw contains **{verdict['struct_leaf_fields']['total_extracted']} "
        "struct leaf fields** (confirmed 31 by assertion)."
    ),
    "",
    "### Field inventory",
    "",
    "| Path | Leaf Fields |",
    "|------|-------------|",
    "| `details.*` | gameSpeed, isBlizzardMap, timeUTC |",
    "| `header.*` | elapsedGameLoops, version |",
    "| `initData.gameDescription.gameOptions.*` | advancedSharedControl, amm, battleNet, clientDebugFlags, competitive, cooperative, fog, heroDuplicatesAllowed, lockTeams, noVictoryOrDefeat, observers, practice, randomRaces, teamsTogether, userDifficulty |",
    "| `initData.gameDescription.*` | gameSpeed, isBlizzardMap, mapAuthorName, mapFileSyncChecksum, mapSizeX, mapSizeY, maxPlayers |",
    "| `metadata.*` | baseBuild, dataBuild, gameVersion, mapName |",
    "",
    "---",
    "",
    "## Map Name Utility (T02)",
    "",
    f"- Distinct map names in `metadata.mapName`: {verdict['map_name_analysis']['meta_map_count']}",
    f"- Null count: {verdict['map_name_analysis']['meta_mapName_nulls']}",
    f"- In foreign_name column of map_aliases_raw: {verdict['map_name_analysis']['in_foreign_names']}",
    f"- In english_name column of map_aliases_raw: {verdict['map_name_analysis']['in_english_names']}",
    f"- In neither: {verdict['map_name_analysis']['in_neither']}",
    f"- Alias pairs with foreign==english: {verdict['map_name_analysis']['alias_identity_pct']:.1f}%",
    "",
    "---",
    "",
    "## Event View Temporal Characterization (T03)",
    "",
    f"- `game_events_raw`: {GAME_EVENTS_ROW_COUNT:,} rows (from schema YAML; COUNT would timeout)",
    f"- `tracker_events_raw`: {verdict['event_row_counts']['tracker_events_raw']:,} rows (live COUNT)",
    f"- `message_events_raw`: {verdict['event_row_counts']['message_events_raw']:,} rows (live COUNT)",
    "",
    f"Tracker events min loop: **{verdict['tracker_events_loop_range']['min_loop']}**",
    f"Tracker events max loop: {verdict['tracker_events_loop_range']['max_loop']:,}",
    "",
    "Loop=0 events in both game_events_raw and tracker_events_raw include initialization",
    "events (PlayerSetup, UnitBorn). These are game-initialization artifacts, not pre-game",
    "information. All events are **IN_GAME** per Invariant #3.",
    "",
    "---",
    "",
    "## Table Utility Verdicts",
    "",
    "| Table/View | Verdict | I3 Classification |",
    "|------------|---------|-------------------|",
]

for tbl, v in verdict["table_verdicts"].items():
    i3 = v.get("i3_class", v.get("i3_classes", ""))
    if isinstance(i3, list):
        i3 = "; ".join(i3)
    md_lines.append(f"| `{tbl}` | {_verdict_badge(v['verdict'])} | {i3} |")

md_lines += [
    "",
    "---",
    "",
    "## SQL Queries (Invariant #6)",
    "",
]

for qname, qsql in sql_queries.items():
    md_lines.append(f"### `{qname}`")
    md_lines.append("")
    md_lines.append("```sql")
    md_lines.append(qsql.strip())
    md_lines.append("```")
    md_lines.append("")

output_md = artifact_dir / "01_03_03_table_utility.md"
with open(output_md, "w") as f:
    f.write("\n".join(md_lines))
print(f"Saved: {output_md}")

print("\n=== Step 01_03_03 complete ===")
print(f"Artifacts:\n  {output_json}\n  {output_md}")

Saved: /Users/tomaszpionka/Projects/rts-outcome-prediction/src/rts_predict/games/sc2/datasets/sc2egset/reports/artifacts/01_exploration/03_profiling/01_03_03_table_utility.md

=== Step 01_03_03 complete ===
Artifacts:
  /Users/tomaszpionka/Projects/rts-outcome-prediction/src/rts_predict/games/sc2/datasets/sc2egset/reports/artifacts/01_exploration/03_profiling/01_03_03_table_utility.json
  /Users/tomaszpionka/Projects/rts-outcome-prediction/src/rts_predict/games/sc2/datasets/sc2egset/reports/artifacts/01_exploration/03_profiling/01_03_03_table_utility.md
